# Building LLM Applications: Agents, Memory & RAG
### Complete Course Notebook

---

## 📋 Table of Contents
- **Day 1**
  - [Session 1: Prompt Reliability Patterns](#session-1)
  - [Session 2: Local vs Cloud Models + Ollama Setup](#session-2)
  - [Session 3: Tool Calling + Agent with FastAPI](#session-3)
- **Day 2**
  - [Session 4: RAG with LangChain + ChromaDB](#session-4)
  - [Session 5: Orchestration Patterns](#session-5)
  - [Session 6: Tracing + Evals with LangSmith](#session-6)

## ⚙️ Prerequisites & Installation

Run this cell first to install all required packages.

In [ ]:
# In terminal
# python -m venv venv
# venv\Scripts\activate 

# pip install ipykernel
# python -m ipykernel install --user --name=venv --display-name "Python (venv)"

# In your notebook: Kernel → Change Kernel → Python (venv)

In [ ]:
# Install all dependencies
!pip install fastapi uvicorn langchain langchain-ollama langchain-community \
             langchain-text-splitters langchainhub python-dotenv \
             chromadb pypdf langsmith pydantic langgraph 

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.41.5-cp314-cp314-win_amd64.whl.metadata (7.4 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl.metadata (2.4 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.11-py3-none-any.wh

Could not find platform independent libraries <prefix>

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 🦙 Ollama Setup (run these in your Terminal, not here)

```bash
# 1. Download Ollama from https://ollama.com (macOS, Windows, Linux)

# 2. Verify installation
ollama --version

# 3. Pull models
ollama pull qwen2.5:1.5b # light model, user friendly
ollama pull nomic-embed-text   # for RAG embeddings

# 4. Start the Ollama server
ollama serve
```

---
# 📅 DAY 1

## Session 1: Prompt Reliability Patterns

### Key Concepts
| Term | Description |
|------|-------------|
| **Context Window** | Max tokens the model can "see" (Claude Sonnet ~200k, GPT-4o ~128k) |
| **Tokens** | Units of text the LLM processes (words/subwords) |
| **Temperature** | Controls randomness: 0 = deterministic, >1.5 = chaotic |
| **Latency** | How long you wait — longer output = longer wait |
| **Cost** | Priced per 1M tokens in + out |

### The 4 Prompt Reliability Patterns

In [1]:
# ── Pattern 1: Give it a Job Title, Not Just a Task ──────────────────────────

bad_prompt_1 = "Answer the user's coding questions."

good_prompt_1 = """
You are a senior backend engineer specialised in Python and PostgreSQL,
helping a team of mid-level developers during a code review session.
"""

# ── Pattern 2: Rules Are Better Than Vibes ───────────────────────────────────

bad_prompt_2 = "Be professional and don't answer irrelevant questions."

good_prompt_2 = """
RULES
1. Only answer questions about Python, SQL, or system design.
2. If the user asks about anything else, say exactly: "That's outside my scope here."
3. Always show code before explaining it.
4. Maximum response length: 300 words.
5. Never say "As an AI language model."
"""

# ── Pattern 3: Show It, Don't Just Tell It (Few-shot Examples) ───────────────

good_prompt_3 = """
EXAMPLE OF A GOOD RESPONSE
User: "What's the difference between a list and a tuple?"
Answer: "Lists are mutable — you can change them after creation.
Tuples are immutable. Use tuples for fixed data, lists when contents will change."

EXAMPLE OF A BAD RESPONSE
User: "What's the difference between a list and a tuple?"
Answer: "Great question! Both lists and tuples are incredibly useful data structures
in Python with some key similarities but also some important differences worth understanding..."
"""

# ── Pattern 4: Tell It What To Do When It's Stuck ────────────────────────────

good_prompt_4 = """
WHEN YOU DON'T KNOW
If you don't have enough information to answer accurately, say:
"I need more context to answer this properly. Could you share [specific thing]?"
Never guess at specifics like variable names, function signatures,
or database schemas you haven't been shown.

WHEN THE QUESTION IS OUT OF SCOPE
Say exactly: "That's outside what I can help with here."
Do not apologise more than once.
Do not offer alternatives outside the defined scope.
"""

print("✅ Prompt patterns defined.")

✅ Prompt patterns defined.


In [2]:
# ── Structured Output with Schemas ───────────────────────────────────────────
# Instead of free text, ask the model to fill in a form (JSON schema).

structured_output_prompt = """
Analyze the following code and return a JSON object with ONLY these fields:
{
  "issues": ["list of issues found"],
  "severity": "low | medium | high | critical",
  "summary": "one-sentence summary",
  "line_numbers": [list of affected line numbers],
  "suggested_fix": "corrected code string"
}

Code to analyze:
cursor.execute(f"SELECT * FROM users WHERE id = {user_id}")
"""

# Expected output:
import json
expected_output = {
    "issues": ["SQL injection: user_id is pasted directly into the query"],
    "severity": "high",
    "summary": "Critical SQL injection on line 1.",
    "line_numbers": [1],
    "suggested_fix": "cursor.execute('SELECT * FROM users WHERE id = %s', (user_id,))"
}
print(json.dumps(expected_output, indent=2))

{
  "issues": [
    "SQL injection: user_id is pasted directly into the query"
  ],
  "severity": "high",
  "summary": "Critical SQL injection on line 1.",
  "line_numbers": [
    1
  ],
  "suggested_fix": "cursor.execute('SELECT * FROM users WHERE id = %s', (user_id,))"
}


---
## Session 2: Local vs Cloud Models + Running with Ollama

| Aspect | Local Model | Cloud Model |
|--------|------------|-------------|
| **Privacy** | More secure, data stays local | Data sent to cloud servers |
| **Cost** | Free after hardware | Pay-per-token |
| **Speed** | No network round-trip (hardware-dependent) | Network overhead |
| **Setup** | Ollama | API Key |

### Best Practices for Local Model Prompting
1. **Role + Task + Format** — always give the model a role, specific task, and exact output shape
2. **Positive instructions** — say what TO do ("Output only code") not what to avoid
3. **Few-shot examples** — add 1–2 input/output examples; highest-ROI technique for local models

In [4]:
# ── Minimal Local Assistant Loop ─────────────────────────────────────────────
# Make sure Ollama is running: `ollama serve` in a terminal

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

def create_local_assistant(model_name="qwen2.5:1.5b", temperature=0.3):
    """Create a simple local chat assistant using Ollama."""
    llm = ChatOllama(model=model_name, temperature=temperature)
    return llm

def chat(llm, user_input, system_prompt=None):
    """Send a message and get a response."""
    messages = []
    if system_prompt:
        messages.append(SystemMessage(content=system_prompt))
    messages.append(HumanMessage(content=user_input))
    response = llm.invoke(messages)
    return response.content

# --- Run it ---
system = """
You are a senior backend engineer specialised in Python and PostgreSQL.
RULES:
1. Only answer questions about Python, SQL, or system design.
2. Always show code before explaining it.
3. Maximum response length: 200 words.
"""

llm = create_local_assistant()
response = chat(llm, "What is a list comprehension?", system_prompt=system)
print(response)

A list comprehension is a concise way to create lists in Python by iterating over elements of an iterable and applying transformations to each element. It allows you to generate a new list based on the results of some operation applied to each item in the original sequence.

Here's how it works:

```python
# Example: Creating a list of squares from 1 to 5
squares = [x**2 for x in range(1, 6)]
print(squares)  # Output: [1, 4, 9, 16, 25]

# Example: Filtering even numbers from a list
evens = [num for num in [10, 23, 45, 87, 2, 5] if num % 2 == 0]
print(evens)  # Output: [10, 45, 87]

# Example: Mapping a function to each element
names = ['Alice', 'Bob', 'Charlie']
greetings = [f"Hello {name}" for name in names]
print(greetings)  # Output: ["Hello Alice", "Hello Bob", "Hello Charlie"]

# Example: Using list comprehension with multiple conditions
filtered_names = [name for name in names if len(name) > 3 and name[0] == 'C']
print(filtered_names)  # Output: ['Charlie']

# List comprehensions

---
## Session 3: Tool Calling + Agent Service

### What makes an "agent"?
An agent combines: **Model** (LLM brain) + **Tools** (real-world actions) + **Prompt** (standing instructions) + **Memory** (persistence across sessions)

### Part A: Standalone Agent Script (`agent_local.py`)

---
## Step 1: Define the Law Knowledge Base & Tool

We create a **structured knowledge base** of laws keyed by category and jurisdiction,
then wrap the lookup logic inside a **LangChain `@tool`** 

The LLM decides *when* to call this tool and *what arguments* to pass based on the claimant profile.


In [5]:
# ── Step 1: Define the Law Information Tool ──────────────────────────────────
import json
from langchain_core.tools import tool

# ── Structured Law Knowledge Base ────────────────────────────────────────────
# In production this would be a database or RAG retrieval — here we use a
# dictionary so the notebook is self-contained.

LAW_DATABASE = {
    "personal_injury": {
        "malaysia": {
            "applicable_laws": [
                "Civil Law Act 1956 (Act 67) — Section 7: Right of action for wrongful act causing death",
                "Civil Law Act 1956 — Section 28A: Assessment of damages for personal injuries",
                "Limitation Act 1953 (Act 254) — Section 6(1): 6-year limitation for tort claims",
                "Employees' Social Security Act 1969 (Act 4) — SOCSO coverage",
            ],
            "limitation_period": "6 years from the date of the incident (Section 6(1), Limitation Act 1953)",
            "remedies": [
                "General damages (pain & suffering)",
                "Special damages (medical bills, loss of earnings)",
                "Future loss of earnings",
                "Loss of amenities of life",
            ],
            "key_precedents": [
                "Ong Ah Long v Dr S Underwood [1983] — Lump sum damages assessment",
                "Chan Chin Ming v Lim Yok Eng [1994] — Multiplier-multiplicand approach",
            ],
        },
        "singapore": {
            "applicable_laws": [
                "Work Injury Compensation Act (WICA) Cap 354",
                "Limitation Act (Cap 163) — Section 24A: 3-year limitation for personal injury",
                "Motor Vehicles (Third-Party Risks & Compensation) Act",
            ],
            "limitation_period": "3 years from the date of injury (Section 24A, Limitation Act)",
            "remedies": [
                "Damages for pain and suffering",
                "Loss of future earnings / earning capacity",
                "Medical and rehabilitation expenses",
            ],
            "key_precedents": [
                "Lai Wai Keong Eugene v Loo Wei Yen [2014] — Guidelines on damages",
            ],
        },
    },
    "employment_dispute": {
        "malaysia": {
            "applicable_laws": [
                "Employment Act 1955 (Act 265) — Covers employees earning ≤ RM4,000/month",
                "Industrial Relations Act 1967 (Act 177) — Unfair dismissal under Section 20",
                "Minimum Wages Order 2022",
                "Employment (Amendment) Act 2022 — Extended coverage, anti-discrimination",
            ],
            "limitation_period": "60 days to file unfair dismissal complaint (Section 20, IRA 1967)",
            "remedies": [
                "Reinstatement to former position",
                "Back wages (max 24 months)",
                "Compensation in lieu of reinstatement",
            ],
            "key_precedents": [
                "Milan Auto Sdn Bhd v Wong Seh Yen [1995] — Employer burden of proof",
            ],
        },
        "singapore": {
            "applicable_laws": [
                "Employment Act (Cap 91)",
                "Employment Claims Act 2016 — Claims via Employment Claims Tribunal",
                "Retirement and Re-employment Act (Cap 274A)",
            ],
            "limitation_period": "1 year from dispute for ECT claims; 6 months for salary claims",
            "remedies": [
                "Monetary compensation up to SGD 20,000 (ECT)",
                "Mediation via TADM before tribunal",
            ],
            "key_precedents": [],
        },
    },
    "motor_vehicle_accident": {
        "malaysia": {
            "applicable_laws": [
                "Road Transport Act 1987 (Act 333)",
                "Civil Law Act 1956 — Section 28A: Damages for personal injury",
                "Motor Vehicles (Third-Party Risks) Act — Compulsory insurance",
            ],
            "limitation_period": "6 years for civil claims; 3 years for fatal accident claims (Section 7, CLA 1956)",
            "remedies": [
                "Third-party insurance claims",
                "General and special damages",
                "Dependency claims (fatal accidents)",
            ],
            "key_precedents": [
                "Takong Tabari v Government of Sarawak [1998] — Dependency claims",
            ],
        },
    },
    "medical_negligence": {
        "malaysia": {
            "applicable_laws": [
                "Civil Law Act 1956 — General negligence framework",
                "Private Healthcare Facilities and Services Act 1998",
                "Medical Act 1971 — Professional standards",
            ],
            "limitation_period": "6 years from date of negligent act (Limitation Act 1953)",
            "remedies": [
                "Compensatory damages",
                "Aggravated damages in severe cases",
                "Complaint to Malaysian Medical Council",
            ],
            "key_precedents": [
                "Foo Fio Na v Dr Soo Fook Mun [2007] — Informed consent standard",
            ],
        },
    },
}


@tool
def extract_law_info(category: str, jurisdiction: str = "malaysia") -> str:
    """
    Extracts relevant law information based on an incident category and jurisdiction.

    Use this tool when you receive a claimant's profile and need to find applicable laws,
    limitation periods, remedies, and legal precedents.

    Args:
        category: The type of legal matter. Must be one of:
                  'personal_injury', 'employment_dispute',
                  'motor_vehicle_accident', 'medical_negligence'
        jurisdiction: The country/jurisdiction. Currently supports:
                      'malaysia', 'singapore'

    Returns:
        JSON string with applicable_laws, limitation_period, remedies, and key_precedents.
    """
    category = category.lower().strip().replace(" ", "_")
    jurisdiction = jurisdiction.lower().strip()

    if category not in LAW_DATABASE:
        available = ", ".join(LAW_DATABASE.keys())
        return json.dumps({
            "error": f"Category '{category}' not found. Available: {available}"
        })

    if jurisdiction not in LAW_DATABASE[category]:
        available = ", ".join(LAW_DATABASE[category].keys())
        return json.dumps({
            "error": f"Jurisdiction '{jurisdiction}' not found for '{category}'. Available: {available}"
        })

    result = LAW_DATABASE[category][jurisdiction]
    return json.dumps(result, indent=2)


# Verify the tool
tools = [extract_law_info]

print("✅ Law information tool defined.")
print(f"   Tool name       : {extract_law_info.name}")
print(f"   Tool description: {extract_law_info.description[:80]}...")
print(f"\n📚 Knowledge base covers:")
for cat in LAW_DATABASE:
    jurisdictions = ", ".join(LAW_DATABASE[cat].keys())
    print(f"   • {cat} → [{jurisdictions}]")


✅ Law information tool defined.
   Tool name       : extract_law_info
   Tool description: Extracts relevant law information based on an incident category and jurisdiction...

📚 Knowledge base covers:
   • personal_injury → [malaysia, singapore]
   • employment_dispute → [malaysia, singapore]
   • motor_vehicle_accident → [malaysia]
   • medical_negligence → [malaysia]


---
## Step 2: Set Up the Agent LLM + Build the ReAct Agent

We use `ChatOllama` and `create_react_agent`.


In [6]:
# ── Step 2: Set Up the Agent ──────────────────────────────────────────────────
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, SystemMessage

SYSTEM_PROMPT = """You are a legal research assistant specializing in Malaysian and Singaporean law.

When a user provides a CLAIMANT PROFILE, you must:
1. Identify the incident category (personal_injury, employment_dispute, motor_vehicle_accident, medical_negligence).
2. Identify the jurisdiction from the profile.
3. Use the extract_law_info tool to retrieve applicable laws.
4. Present a clear, structured legal summary covering:
   - Applicable laws and statutes
   - Limitation period (how long the claimant has to file)
   - Available legal remedies
   - Relevant case precedents

Always use the tool — do NOT guess at legal information from memory.
If the profile is ambiguous, extract the most likely category and explain your reasoning.
"""

def get_law_agent(model_name="qwen2.5:1.5b", temperature=0):
    """Build the law extraction agent."""
    llm = ChatOllama(model=model_name, temperature=temperature)
    agent = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)
    print(f"✅ Law Agent created with model: {model_name}")
    return agent

def run_law_agent(agent, claimant_profile: str):
    """Run the agent with a claimant profile."""
    print(f"\n{'='*70}")
    print("📋 CLAIMANT PROFILE:")
    print(claimant_profile)
    print(f"{'='*70}")

    response = agent.invoke({
        "messages": [HumanMessage(content=claimant_profile)]
    })

    answer = response["messages"][-1].content

    # Log tool usage
    for msg in response["messages"]:
        if hasattr(msg, "name") and msg.name == "extract_law_info":
            print(f"\n🔧 Tool called: {msg.name}")
            print(f"   Result preview: {str(msg.content)[:200]}...")

    print(f"\n📑 LEGAL ANALYSIS:")
    print(answer)
    return answer

print("✅ Agent builder functions defined.")

✅ Agent builder functions defined.


---
## Step 3: Test the Agent with Sample Claimant Profiles

Let's test with different types of claimant profiles to see the agent pick the right category and call the tool.

In [7]:
# ── Step 3: Run the Agent Pipeline ────────────────────────────────────────────
# Make sure Ollama is running: ollama serve

# 1. Build the agent
law_agent = get_law_agent(model_name="qwen2.5:1.5b")

# ── Test Case 1: Personal Injury ─────────────────────────────────────────────
profile_1 = """
CLAIMANT PROFILE
Name        : Ahmad bin Ismail
Age         : 34
Jurisdiction: Malaysia
Incident    : Slipped and fell at a shopping mall due to wet floor with no warning signs.
              Suffered a fractured wrist and herniated disc.
Date        : 15 March 2024
Employment  : Private sector employee, monthly income RM 5,500
Medical cost: RM 23,000 to date
Status      : Still undergoing physiotherapy
"""
run_law_agent(law_agent, profile_1)


C:\Users\ALPHV\AppData\Local\Temp\ipykernel_25080\4273696186.py:25: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)


✅ Law Agent created with model: qwen2.5:1.5b

📋 CLAIMANT PROFILE:

CLAIMANT PROFILE
Name        : Ahmad bin Ismail
Age         : 34
Jurisdiction: Malaysia
Incident    : Slipped and fell at a shopping mall due to wet floor with no warning signs.
              Suffered a fractured wrist and herniated disc.
Date        : 15 March 2024
Employment  : Private sector employee, monthly income RM 5,500
Medical cost: RM 23,000 to date
Status      : Still undergoing physiotherapy


🔧 Tool called: extract_law_info
   Result preview: {
  "applicable_laws": [
    "Civil Law Act 1956 (Act 67) \u2014 Section 7: Right of action for wrongful act causing death",
    "Civil Law Act 1956 \u2014 Section 28A: Assessment of damages for perso...

📑 LEGAL ANALYSIS:
### Legal Summary

#### Applicable Laws and Statutes
- **Civil Law Act 1956 (Act 67)**: Section 7: Right of action for wrongful act causing death.
- **Civil Law Act 1956**: Section 28A: Assessment of damages for personal injuries.
- **Limitation Act 

"### Legal Summary\n\n#### Applicable Laws and Statutes\n- **Civil Law Act 1956 (Act 67)**: Section 7: Right of action for wrongful act causing death.\n- **Civil Law Act 1956**: Section 28A: Assessment of damages for personal injuries.\n- **Limitation Act 1953 (Act 254)**: Section 6(1): 6-year limitation for tort claims.\n\n#### Limitation Period\nThe claimant has up to 6 years from the date of the incident (March 15, 2024) to file a lawsuit. This period is governed by the **Limitation Act 1953**.\n\n#### Available Legal Remedies\n- **General Damages**: Compensation for pain and suffering.\n- **Special Damages**: Medical bills, loss of earnings.\n- **Future Loss of Earnings**: Compensation for anticipated future income losses.\n- **Loss of Amenities of Life**: Compensation for the value of lost life expectancy or quality of life.\n\n#### Relevant Case Precedents\n- **Ong Ah Long v Dr S Underwood [1983]**: Lump sum damages assessment.\n- **Chan Chin Ming v Lim Yok Eng [1994]**: Multipli

In [8]:
# ── Test Case 2: Employment Dispute ───────────────────────────────────────────
profile_2 = """
CLAIMANT PROFILE
Name        : Siti Nurhaliza binti Razak
Age         : 28
Jurisdiction: Malaysia
Incident    : Terminated from employment without notice or valid reason after
              reporting safety violations to management.
Date        : 2 January 2025
Employment  : Factory operator, monthly salary RM 2,200, employed for 4 years
Status      : Currently unemployed, seeking legal recourse
"""
run_law_agent(law_agent, profile_2)



📋 CLAIMANT PROFILE:

CLAIMANT PROFILE
Name        : Siti Nurhaliza binti Razak
Age         : 28
Jurisdiction: Malaysia
Incident    : Terminated from employment without notice or valid reason after
              reporting safety violations to management.
Date        : 2 January 2025
Employment  : Factory operator, monthly salary RM 2,200, employed for 4 years
Status      : Currently unemployed, seeking legal recourse


🔧 Tool called: extract_law_info
   Result preview: {
  "applicable_laws": [
    "Employment Act 1955 (Act 265) \u2014 Covers employees earning \u2264 RM4,000/month",
    "Industrial Relations Act 1967 (Act 177) \u2014 Unfair dismissal under Section 20...

📑 LEGAL ANALYSIS:
### Legal Summary

**Applicable Laws and Statutes:**
- **Employment Act 1955 (Act 265)**: Covers employees earning up to RM4,000/month.
- **Industrial Relations Act 1967 (Act 177)**: Unfair dismissal under Section 20.
- **Minimum Wages Order 2022**: Ensures fair wages and working conditions.
- **Employ

"### Legal Summary\n\n**Applicable Laws and Statutes:**\n- **Employment Act 1955 (Act 265)**: Covers employees earning up to RM4,000/month.\n- **Industrial Relations Act 1967 (Act 177)**: Unfair dismissal under Section 20.\n- **Minimum Wages Order 2022**: Ensures fair wages and working conditions.\n- **Employment (Amendment) Act 2022**: Extended coverage, anti-discrimination measures.\n\n**Limitation Period:**\nThe claimant has up to 60 days from the date of termination to file a complaint under Section 20 of the Industrial Relations Act 1967. This period is crucial for initiating legal proceedings and ensuring timely resolution.\n\n**Available Legal Remedies:**\n- **Reinstatement to Former Position**: The employer must reinstate the claimant to their previous position.\n- **Back Wages (Max 24 Months)**: Claimants can seek back wages, with a maximum of 24 months from the date of termination.\n- **Compensation in Lieu of Reinstatement**: If reinstatement is not possible, compensation ma

In [9]:
# ── Test Case 3: Motor Vehicle Accident ───────────────────────────────────────
profile_3 = """
CLAIMANT PROFILE
Name        : Raj Kumar a/l Krishnan
Age         : 45
Jurisdiction: Malaysia
Incident    : Rear-ended by a lorry while stationary at traffic light.
              Sustained whiplash injury and damage to vehicle.
Date        : 10 August 2023
Employment  : Grab driver, average monthly income RM 4,000
Medical cost: RM 8,500
Vehicle cost: RM 15,000 repair estimate
Status      : Unable to work for 3 months
"""
run_law_agent(law_agent, profile_3)



📋 CLAIMANT PROFILE:

CLAIMANT PROFILE
Name        : Raj Kumar a/l Krishnan
Age         : 45
Jurisdiction: Malaysia
Incident    : Rear-ended by a lorry while stationary at traffic light.
              Sustained whiplash injury and damage to vehicle.
Date        : 10 August 2023
Employment  : Grab driver, average monthly income RM 4,000
Medical cost: RM 8,500
Vehicle cost: RM 15,000 repair estimate
Status      : Unable to work for 3 months


🔧 Tool called: extract_law_info
   Result preview: {
  "applicable_laws": [
    "Civil Law Act 1956 (Act 67) \u2014 Section 7: Right of action for wrongful act causing death",
    "Civil Law Act 1956 \u2014 Section 28A: Assessment of damages for perso...

📑 LEGAL ANALYSIS:
### Legal Summary

**Applicable Laws and Statutes:**

- **Civil Law Act 1956 (Act 67)**
  - Section 7: Right of action for wrongful act causing death
  - Section 28A: Assessment of damages for personal injuries

- **Limitation Act 1953 (Act 254)**
  - Section 6(1): 6-year limitati

"### Legal Summary\n\n**Applicable Laws and Statutes:**\n\n- **Civil Law Act 1956 (Act 67)**\n  - Section 7: Right of action for wrongful act causing death\n  - Section 28A: Assessment of damages for personal injuries\n\n- **Limitation Act 1953 (Act 254)**\n  - Section 6(1): 6-year limitation for tort claims\n\n**Limitation Period:**\n\nThe claimant has six years from the date of the incident to file a lawsuit. This period begins on August 10, 2023.\n\n**Available Legal Remedies:**\n\n- **General Damages (pain & suffering)**\n- **Special Damages (medical bills, loss of earnings)**\n- **Future Loss of Earnings**\n- **Loss of Amenities of Life**\n\n**Relevant Case Precedents:**\n\n- **Ong Ah Long v Dr S Underwood [1983]**\n  - Lump sum damages assessment\n- **Chan Chin Ming v Lim Yok Eng [1994]**\n  - Multiplier-multiplicand approach\n\n### Summary of Claimant's Situation:\n\nRaj Kumar a/l Krishnan, a 45-year-old individual from Malaysia, was rear-ended by a lorry while stationary at a t

---
### Part B: FastAPI Agent API

> **Note:** FastAPI apps can't run directly inside Jupyter. Save the cell below as `law_agent_api.py` and run:
> ```bash
> venv/Scripts/activate
> uvicorn law_agent_api:app --reload
> ```
> Then visit: http://localhost:8000/docs

In [ ]:
"""
Law Information Extraction API — FastAPI + LangChain Tool Calling + Ollama
--------------------------------------------------------------------------
Run:
  1. ollama serve
  2. ollama pull qwen2.5:1.5b
  3. pip install fastapi uvicorn langchain langchain-ollama langgraph pydantic
  4. uvicorn law_agent_api:app --reload
  5. Open http://localhost:8000/docs
"""

import json
from contextlib import asynccontextmanager
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent

# ── Pydantic Schemas ──────────────────────────────────────────────────────────

class ClaimantRequest(BaseModel):
    """Input schema: the claimant's summary profile as free text."""
    profile: str = Field(
        ...,
        min_length=10,
        examples=[
            "Name: Ahmad, Age: 34, Jurisdiction: Malaysia, "
            "Incident: Slipped at mall, fractured wrist. Date: 15 March 2024."
        ],
    )
    model: str = Field(default="qwen2.5:1.5b", examples=["qwen2.5:1.5b"])

class ToolCallLog(BaseModel):
    tool_name: str
    tool_input: dict | None = None
    tool_output: str

class LawExtractionResponse(BaseModel):
    claimant_profile: str
    legal_analysis: str
    model: str
    tools_used: list[ToolCallLog] = []

# ── Law Knowledge Base ────────────────────────────────────────────────────────

LAW_DATABASE = {
    "personal_injury": {
        "malaysia": {
            "applicable_laws": [
                "Civil Law Act 1956 (Act 67) — Section 7: Wrongful act causing death",
                "Civil Law Act 1956 — Section 28A: Damages for personal injuries",
                "Limitation Act 1953 (Act 254) — Section 6(1): 6-year limitation",
                "Employees Social Security Act 1969 (SOCSO)",
            ],
            "limitation_period": "6 years from date of incident",
            "remedies": ["General damages", "Special damages", "Future loss of earnings", "Loss of amenities"],
            "key_precedents": ["Ong Ah Long v Dr S Underwood [1983]", "Chan Chin Ming v Lim Yok Eng [1994]"],
        },
        "singapore": {
            "applicable_laws": [
                "Work Injury Compensation Act (WICA)",
                "Limitation Act (Cap 163) — Section 24A: 3-year limitation",
            ],
            "limitation_period": "3 years from date of injury",
            "remedies": ["Pain and suffering damages", "Loss of future earnings", "Medical expenses"],
            "key_precedents": ["Lai Wai Keong Eugene v Loo Wei Yen [2014]"],
        },
    },
    "employment_dispute": {
        "malaysia": {
            "applicable_laws": [
                "Employment Act 1955 (Act 265)",
                "Industrial Relations Act 1967 — Section 20: Unfair dismissal",
                "Employment (Amendment) Act 2022",
            ],
            "limitation_period": "60 days to file unfair dismissal complaint",
            "remedies": ["Reinstatement", "Back wages (max 24 months)", "Compensation in lieu"],
            "key_precedents": ["Milan Auto Sdn Bhd v Wong Seh Yen [1995]"],
        },
    },
    "motor_vehicle_accident": {
        "malaysia": {
            "applicable_laws": [
                "Road Transport Act 1987 (Act 333)",
                "Civil Law Act 1956 — Section 28A",
                "Motor Vehicles (Third-Party Risks) Act",
            ],
            "limitation_period": "6 years for civil claims; 3 years for fatal accident claims",
            "remedies": ["Third-party insurance claims", "General & special damages", "Dependency claims"],
            "key_precedents": ["Takong Tabari v Government of Sarawak [1998]"],
        },
    },
    "medical_negligence": {
        "malaysia": {
            "applicable_laws": [
                "Civil Law Act 1956",
                "Private Healthcare Facilities and Services Act 1998",
                "Medical Act 1971",
            ],
            "limitation_period": "6 years from date of negligent act",
            "remedies": ["Compensatory damages", "Aggravated damages", "MMC complaint"],
            "key_precedents": ["Foo Fio Na v Dr Soo Fook Mun [2007]"],
        },
    },
}


# ── Custom Tool ───────────────────────────────────────────────────────────────

@tool
def extract_law_info(category: str, jurisdiction: str = "malaysia") -> str:
    """
    Extracts relevant law information for a given incident category and jurisdiction.

    Use this tool when you receive a claimant profile and need to find applicable laws,
    limitation periods, remedies, and legal precedents.

    Args:
        category: One of 'personal_injury', 'employment_dispute',
                  'motor_vehicle_accident', 'medical_negligence'.
        jurisdiction: 'malaysia' or 'singapore'.

    Returns:
        JSON with applicable_laws, limitation_period, remedies, key_precedents.
    """
    category = category.lower().strip().replace(" ", "_")
    jurisdiction = jurisdiction.lower().strip()

    if category not in LAW_DATABASE:
        return json.dumps({"error": f"Unknown category. Available: {list(LAW_DATABASE.keys())}"})
    if jurisdiction not in LAW_DATABASE[category]:
        return json.dumps({"error": f"Unknown jurisdiction. Available: {list(LAW_DATABASE[category].keys())}"})

    return json.dumps(LAW_DATABASE[category][jurisdiction], indent=2)


TOOLS = [extract_law_info]

SYSTEM_PROMPT = """You are a legal research assistant specializing in Malaysian and Singaporean law.

When you receive a CLAIMANT PROFILE, you must:
1. Identify the incident category (personal_injury, employment_dispute, motor_vehicle_accident, medical_negligence).
2. Identify the jurisdiction from the profile.
3. Use the extract_law_info tool to retrieve applicable laws.
4. Present a structured legal summary: applicable laws, limitation period, remedies, and precedents.

Always use the tool. Never guess at legal information."""


# ── Agent Factory ─────────────────────────────────────────────────────────────

def build_law_agent(model_name: str, temperature: float):
    llm = ChatOllama(model=model_name, temperature=temperature)
    agent = create_react_agent(llm, TOOLS, prompt=SYSTEM_PROMPT)
    return agent


# ── FastAPI App ───────────────────────────────────────────────────────────────

default_agent = None

@asynccontextmanager
async def lifespan(app: FastAPI):
    global default_agent
    print("⏳ Building default law agent (qwen2.5:1.5b) ...")
    default_agent = build_law_agent("qwen2.5:1.5b", temperature=0.0)
    print("✅ Law Agent ready.")
    yield
    print("🛑 Shutting down.")


app = FastAPI(
    title="Law Information Extraction API",
    description=(
        "Submit a claimant's summary profile and receive a structured legal analysis "
        "powered by a LangGraph ReAct agent with Ollama."
    ),
    version="1.0.0",
    lifespan=lifespan,
)


# ── Routes ────────────────────────────────────────────────────────────────────

@app.get("/health")
def health_check():
    return {"status": "ok", "tools": [t.name for t in TOOLS]}


@app.post("/law/analyze", response_model=LawExtractionResponse)
def analyze_claimant(req: ClaimantRequest):
    """
    Accepts a claimant summary profile and returns a legal analysis
    with applicable laws, limitation periods, remedies, and precedents.
    """
    temperature: float = 0.0
    if req.model == "qwen2.5:1.5b" and temperature == 0.0 and default_agent:
        agent = default_agent
    else:
        agent = build_law_agent(req.model, temperature)

    try:
        result = agent.invoke({
            "messages": [HumanMessage(content=req.profile)]
        })
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Agent error: {e}")

    # Extract tool call logs
    tools_used = []
    for msg in result["messages"]:
        if hasattr(msg, "name") and msg.name in [t.name for t in TOOLS]:
            tools_used.append(ToolCallLog(
                tool_name=msg.name,
                tool_input=getattr(msg, "tool_input", None),
                tool_output=str(msg.content),
            ))

    answer = result["messages"][-1].content

    return LawExtractionResponse(
        claimant_profile=req.profile,
        legal_analysis=answer,
        model=req.model,
        tools_used=tools_used,
    )


if __name__ == "__main__":
    import uvicorn
    uvicorn.run("law_agent_api:app", host="0.0.0.0", port=8000, reload=True)


# print("✅ law_agent_api.py saved! Run it with:")
# print("   uvicorn law_agent_api:app --reload")

---
# 📅 DAY 2

## Session 4: RAG Grounding with LangChain + ChromaDB

### RAG Fundamentals
| Concept | What It Does |
|---------|-------------|
| **Chunking** | Breaks large docs into smaller pieces for the LLM |
| **Embedding** | Converts text to vectors; similar meanings → close vectors |
| **Retrieval** | Finds top-k chunks most similar to user's query |
| **Citation** | Each chunk carries metadata (source, page) for verifiability |

### Setup: Put your PDF in a `documents/` folder
```
your_project/
├── documents/
│   └── mydocument.pdf   ← your file here
├── chroma_db/           ← auto-created by ChromaDB
```

In [10]:
# ── Step 1: Load Documents ────────────────────────────────────────────────────
import os
from langchain_community.document_loaders import PyPDFLoader

DATA_PATH = "documents/"
PDF_FILENAME = "sample_journal.pdf"  # ← Replace with your PDF filename

def load_documents():
    """Loads documents from the specified data path."""
    pdf_path = os.path.join(DATA_PATH, PDF_FILENAME)
    loader = PyPDFLoader(pdf_path)
    # Alternative: UnstructuredPDFLoader(pdf_path)
    documents = loader.load()
    print(f"✅ Loaded {len(documents)} page(s) from {pdf_path}")
    return documents

documents = load_documents()  # Uncomment after placing your PDF in data/

✅ Loaded 25 page(s) from documents/sample_journal.pdf


In [11]:
# ── Step 2: Split Documents ───────────────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents):
    """Splits documents into smaller chunks suitable for embedding."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,       # Max characters per chunk
        chunk_overlap=200,     # Overlap between chunks to maintain context
        length_function=len,
        is_separator_regex=False,
    )
    all_splits = text_splitter.split_documents(documents)
    print(f"✅ Split into {len(all_splits)} chunks")
    return all_splits

chunks = split_documents(documents)  # Uncomment after loading documents

✅ Split into 114 chunks


In [12]:
# ── Step 3: Configure Embedding Model ─────────────────────────────────────────
# First pull the model in terminal: ollama pull nomic-embed-text

from langchain_ollama import OllamaEmbeddings

def get_embedding_function(model_name="nomic-embed-text"):
    """Initializes the Ollama embedding function."""
    # Ensure Ollama server is running: ollama serve
    embeddings = OllamaEmbeddings(model=model_name)
    print(f"✅ Initialized Ollama embeddings with model: {model_name}")
    return embeddings

embedding_function = get_embedding_function()  # Uncomment to initialize

✅ Initialized Ollama embeddings with model: nomic-embed-text


In [13]:
# ── Step 4: Set Up ChromaDB Vector Store ──────────────────────────────────────
from langchain_community.vectorstores import Chroma

CHROMA_PATH = "chroma_db"  # Directory to store ChromaDB data

def get_vector_store(embedding_function, persist_directory=CHROMA_PATH):
    """Initializes or loads the Chroma vector store."""
    vectorstore = Chroma(
        persist_directory=persist_directory,
        embedding_function=embedding_function
    )
    print(f"✅ Vector store initialized/loaded from: {persist_directory}")
    return vectorstore

def index_documents(chunks, embedding_function, persist_directory=CHROMA_PATH):
    """Indexes document chunks into the Chroma vector store."""
    print(f"Indexing {len(chunks)} chunks...")
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_function,
        persist_directory=persist_directory
    )
    vectorstore.persist()  # Save to disk
    print(f"✅ Indexing complete. Data saved to: {persist_directory}")
    return vectorstore

# Load existing DB (after initial indexing):
embedding_function = get_embedding_function()
vector_store = Chroma(persist_directory=CHROMA_PATH, embedding_function=embedding_function)

print("✅ Vector store functions defined.")

✅ Initialized Ollama embeddings with model: nomic-embed-text


C:\Users\ALPHV\AppData\Local\Temp\ipykernel_25080\617273936.py:29: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(persist_directory=CHROMA_PATH, embedding_function=embedding_function)


✅ Vector store functions defined.


In [14]:
# ── Step 5: Build the RAG Chain ───────────────────────────────────────────────
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def create_rag_chain(vector_store, llm_model_name="qwen2.5:1.5b", context_window=8192):
    """Creates the RAG chain using LangChain Expression Language (LCEL)."""

    # Initialize the LLM
    llm = ChatOllama(
        model=llm_model_name,
        temperature=0,           # Lower temp = more factual RAG answers
        num_ctx=context_window   # IMPORTANT: set context window
    )
    print(f"✅ Initialized ChatOllama with model: {llm_model_name}, ctx: {context_window}")

    # Create the retriever
    retriever = vector_store.as_retriever(
        search_type="similarity",  # Or "mmr" for diversity
        search_kwargs={'k': 3}     # Retrieve top 3 relevant chunks
    )
    print("✅ Retriever initialized.")

    # Define the prompt template
    template = """Answer the question based ONLY on the following context:

{context}

Question: {question}
"""
    prompt = ChatPromptTemplate.from_template(template)
    print("✅ Prompt template created.")

    # Build the LCEL RAG chain: retrieve → prompt → LLM → parse
    rag_chain = (
        {"context": retriever, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    print("✅ RAG chain created.")
    return rag_chain

# ── Step 6: Query Your Documents ─────────────────────────────────────────────
def query_rag(chain, question):
    """Queries the RAG chain and prints the response."""
    print(f"\nQuestion: {question}")
    response = chain.invoke(question)
    print("\nResponse:")
    print(response)
    return response

print("✅ RAG chain functions defined.")

✅ RAG chain functions defined.


In [16]:
# ── Full RAG Pipeline (run after placing your PDF in data/) ──────────────────
# Make sure: ollama serve is running + nomic-embed-text is pulled

# 1. Load Documents
docs = load_documents()

# 2. Split
chunks = split_documents(docs)

# 3. Embed
embedding_function = get_embedding_function()

# 4. Index (only needed once — comment out after first run)
vector_store = index_documents(chunks, embedding_function)
# To reload an existing index instead:
# vector_store = Chroma(persist_directory=CHROMA_PATH, embedding_function=embedding_function)

# 5. Build RAG Chain
rag_chain = create_rag_chain(vector_store, llm_model_name="qwen2.5:1.5b")

# 6. Query
query_rag(rag_chain, "What is the main topic of the document?")

✅ Loaded 25 page(s) from documents/sample_journal.pdf
✅ Split into 114 chunks
✅ Initialized Ollama embeddings with model: nomic-embed-text
Indexing 114 chunks...
✅ Indexing complete. Data saved to: chroma_db
✅ Initialized ChatOllama with model: qwen2.5:1.5b, ctx: 8192
✅ Retriever initialized.
✅ Prompt template created.
✅ RAG chain created.

Question: What is the main topic of the document?

Response:
The main topic of the document appears to be an "AMFormer-based framework for accident responsibility attribution" and its use in providing an interpretable analysis with traffic accident features. The document likely discusses how this framework can help understand and explain the reasons behind different levels of responsibility in traffic accidents, making it easier to interpret and analyze.


'The main topic of the document appears to be an "AMFormer-based framework for accident responsibility attribution" and its use in providing an interpretable analysis with traffic accident features. The document likely discusses how this framework can help understand and explain the reasons behind different levels of responsibility in traffic accidents, making it easier to interpret and analyze.'

---
## Session 5: Orchestration Patterns

| Pattern | Control Flow | Best For |
|---------|-------------|----------|
| **Chain** | Linear, deterministic | ETL, Summarize |
| **Router** | Conditional branching | Multi-domain assistants |
| **Planner** | Dynamic branching | Complex research tasks |
| **Tool Loop** | Interactive, self-directed | Agentic tool-use tasks |

### Memory: Session State vs Durable Memory

**Rule:** *Does this need to survive beyond this conversation?*
- Yes → persist it (vector store / DB)
- No → keep it ephemeral in context

In [17]:
# ── Multi-Step Workflow: Retrieve → Decide → Act → Format ────────────────────
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen2.5:1.5b", temperature=0)

# Step 1 - Retrieve (reuse rag_chain retriever from Session 4)
# Step 2 - Decide: classify the query intent
classify_prompt = ChatPromptTemplate.from_template("""
Given this user question, classify the intent as one of:
[summarize, compare, explain, other]

Question: {question}
Intent (one word only):
""")

classify_chain = classify_prompt | llm | StrOutputParser()

# Step 3 - Act: route to different response formats based on intent
format_prompt = ChatPromptTemplate.from_template("""
Context: {context}
Question: {question}
Intent: {intent}

Based on the intent, format your answer appropriately:
- summarize: bullet points
- compare: a table
- explain: step-by-step
- other: plain text

Answer:
""")

format_chain = format_prompt | llm | StrOutputParser()

print("✅ Orchestration chain defined.")
print("Use classify_chain.invoke({'question': '...'}) to test classification.")

✅ Orchestration chain defined.
Use classify_chain.invoke({'question': '...'}) to test classification.


In [18]:
classify_chain.invoke({'question': 'Help me rearrange'})

'explain'

---
## Session 6: Tracing + Evaluations with LangSmith

### Setup LangSmith
```bash
pip install langsmith
```

Set environment variables (create a `.env` file):
```
LANGCHAIN_TRACING_V2=true
LANGCHAIN_API_KEY=your_langsmith_api_key
LANGCHAIN_PROJECT=my-rag-project
```

In [ ]:
# ── Step 1: Loads.env ────────────────────────
from dotenv import load_dotenv
load_dotenv()  # Loads LANGCHAIN_TRACING_V2 and LANGCHAIN_API_KEY from .env

True

In [24]:
# ── Step 2: Create Golden Dataset & Run Evals ────────────────────────
# Add this import and decorator to enable automatic tracing in LangSmithfrom langsmith import Client, evaluate, traceable
from langsmith import Client, evaluate, traceable
from langchain_core.messages import HumanMessage


def run_law_evals(law_agent):
    """
    Creates a golden evaluation dataset for the law extraction agent
    and runs evaluations using the Session 6 pattern.
    """
    client = Client()
    dataset_name = "law-extraction-golden-set"

    # ── 1. Create the golden dataset (For ground truth testing) ─────────────────────────────
    if not any(d.name == dataset_name for d in client.list_datasets()):
        dataset = client.create_dataset(dataset_name)
        client.create_examples(
            inputs=[
                {
                    "profile": (
                        "CLAIMANT PROFILE\n"
                        "Name: Ahmad bin Ismail, Age: 34\n"
                        "Jurisdiction: Malaysia\n"
                        "Incident: Slipped and fell at a shopping mall due to wet floor. "
                        "Fractured wrist and herniated disc.\n"
                        "Date: 15 March 2024\n"
                        "Medical cost: RM 23,000"
                    )
                },
                {
                    "profile": (
                        "CLAIMANT PROFILE\n"
                        "Name: Siti Nurhaliza binti Razak, Age: 28\n"
                        "Jurisdiction: Malaysia\n"
                        "Incident: Terminated without notice or valid reason after "
                        "reporting safety violations.\n"
                        "Date: 2 January 2025\n"
                        "Employment: Factory operator, RM 2,200/month, 4 years tenure"
                    )
                },
                {
                    "profile": (
                        "CLAIMANT PROFILE\n"
                        "Name: Raj Kumar, Age: 45\n"
                        "Jurisdiction: Malaysia\n"
                        "Incident: Rear-ended by a lorry at a traffic light. "
                        "Whiplash injury and vehicle damage.\n"
                        "Date: 10 August 2023"
                    )
                },
                {
                    "profile": (
                        "CLAIMANT PROFILE\n"
                        "Name: Lee Mei Ling, Age: 52\n"
                        "Jurisdiction: Malaysia\n"
                        "Incident: Wrong medication prescribed by doctor leading to "
                        "severe allergic reaction and hospitalization.\n"
                        "Date: 5 June 2024"
                    )
                },
                {
                    "profile": (
                        "CLAIMANT PROFILE\n"
                        "Name: David Tan, Age: 30\n"
                        "Jurisdiction: Singapore\n"
                        "Incident: Injured at construction site due to inadequate "
                        "safety equipment. Broken leg.\n"
                        "Date: 20 November 2024"
                    )
                },
            ],
            outputs=[
                {
                    "answer": "Civil Law Act 1956",
                    "expected_category": "personal_injury",
                    "expected_limitation": "6 years",
                },
                {
                    "answer": "Industrial Relations Act 1967",
                    "expected_category": "employment_dispute",
                    "expected_limitation": "60 days",
                },
                {
                    "answer": "Road Transport Act 1987",
                    "expected_category": "motor_vehicle_accident",
                    "expected_limitation": "6 years",
                },
                {
                    "answer": "Medical Act 1971",
                    "expected_category": "medical_negligence",
                    "expected_limitation": "6 years",
                },
                {
                    "answer": "Work Injury Compensation Act",
                    "expected_category": "personal_injury",
                    "expected_limitation": "3 years",
                },
            ],
            dataset_id=dataset.id,
        )
        print(f"✅ Created golden dataset '{dataset_name}' with 5 examples.")
    else:
        print(f"✅ Dataset '{dataset_name}' already exists.")

    # ── 2. Define the runner ─────────────────────────────────────────────────
    @traceable(name="law-extraction-eval")
    def run_law_extraction(inputs):
        """Runs the law agent on a claimant profile. Returns the output."""
        response = law_agent.invoke({
            "messages": [HumanMessage(content=inputs["profile"])]
        })
        output = response["messages"][-1].content
        return {"output": output}

    # ── 3. Define evaluators ─────────────────────────────────────────────────

    def law_correctness(outputs, reference_outputs):
        expected = reference_outputs["answer"].lower()
        actual = outputs["output"].lower()
        return {
            "key": "law_correctness",
            "score": 1 if expected in actual else 0,
        }

    def limitation_correctness(outputs, reference_outputs):
        expected = reference_outputs["expected_limitation"].lower()
        actual = outputs["output"].lower()
        return {
            "key": "limitation_correctness",
            "score": 1 if expected in actual else 0,
        }

    def category_correctness(outputs, reference_outputs):
        expected_cat = reference_outputs["expected_category"].replace("_", " ")
        actual = outputs["output"].lower()
        return {
            "key": "category_correctness",
            "score": 1 if expected_cat in actual else 0,
        }

    # ── 4. Run the evaluation ────────────────────────────────────────────────
    print("\n⏳ Running evaluations...")
    results = evaluate(
        run_law_extraction,
        data=dataset_name,
        evaluators=[law_correctness, limitation_correctness, category_correctness],
        experiment_prefix="law-extraction-qwen3",
    )
    print("\n📊 Evaluation Results:")
    print(results)
    return results


# ── Run it ──
results = run_law_evals(law_agent)

✅ Dataset 'law-extraction-golden-set' already exists.

⏳ Running evaluations...


c:\Users\ALPHV\Desktop\llm-course\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'law-extraction-qwen3-5ed436c7' at:
https://smith.langchain.com/o/17c527bb-4bcc-41cf-aa53-e78a68a4cb8f/datasets/c248f956-0dbc-4711-acf7-755d0ddc7b7e/compare?selectedSessions=f4cea29b-dbb2-428a-8115-4136a7440d25




5it [02:09, 25.81s/it]


📊 Evaluation Results:
<ExperimentResults law-extraction-qwen3-5ed436c7>
